In [10]:
import math
import random
from typing import Callable, List, Tuple, Dict, Any

import numpy as np
from sklearn.preprocessing import StandardScaler

# If you want tree SCM, you need xgboost installed and working.
from xgboost import XGBRegressor


def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)


In [11]:
def tanh(x): return np.tanh(x)

def leaky_relu(x, negative_slope: float = 0.01):
    return np.where(x > 0, x, negative_slope * x)

def elu(x, alpha: float = 1.0):
    return np.where(x > 0, x, alpha * (np.exp(x) - 1))

def identity(x): return x
def relu(x): return np.maximum(0.0, x)
def relu6(x): return np.minimum(6.0, np.maximum(0.0, x))

def selu(x, lambda_: float = 1.0507, alpha: float = 1.67326):
    return lambda_ * np.where(x > 0, x, alpha * (np.exp(x) - 1))

def silu(x): return x / (1.0 + np.exp(-x))

def softplus(x):
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0)

def hardtanh(x, min_val: float = -1.0, max_val: float = 1.0):
    return np.clip(x, min_val, max_val)

def signum(x): return np.where(x >= 0.0, 1.0, -1.0)
def sine(x): return np.sin(x)
def rbf(x): return np.exp(-x ** 2)
def exponential(x): return np.exp(x)
def sqrt_abs(x): return np.sqrt(np.abs(x))
def unit_interval_indicator(x): return (np.abs(x) <= 1.0).astype(float)
def square(x): return x ** 2
def absolute(x): return np.abs(x)


def random_fourier_activation(x: np.ndarray, n_frequencies: int = 256) -> np.ndarray:
    """
    Random Fourier feature based activation.
    Produces smooth random functions similar to GP draws.
    """
    x_std = (x - np.mean(x, axis=0, keepdims=True)) / (np.std(x, axis=0, keepdims=True) + 1e-6)

    a = np.random.uniform(0, n_frequencies, size=(n_frequencies, 1))
    b = np.random.uniform(0.0, 2 * np.pi, size=(n_frequencies, 1))

    u = np.random.uniform(0.7, 3.0, size=(n_frequencies, 1))
    w = np.exp(-np.exp(u))
    w = w / np.linalg.norm(w)

    # IMPORTANT: broadcast carefully.
    # x_std: (n, d)
    # We want sines: (n, d, nf)
    # Create (n, d, 1) and (1, 1, nf) shapes.
    x3 = x_std[:, :, None]                         # (n, d, 1)
    a3 = a[None, None, :, 0]                        # (1, 1, nf)
    b3 = b[None, None, :, 0]                        # (1, 1, nf)
    sines = np.sin(a3 * x3 + b3)                    # (n, d, nf)

    w3 = w[None, None, :, 0]                        # (1, 1, nf)
    return np.sum(w3 * sines, axis=2)               # (n, d)


def sample_activation() -> Callable[[np.ndarray], np.ndarray]:
    """
    Sample an activation function (TabICL-like menu).
    Random Fourier is sampled 10x more often.
    """
    activations = [
        tanh,
        lambda x: leaky_relu(x, 0.01),
        elu,
        identity,
        relu,
        relu6,
        selu,
        silu,
        softplus,
        hardtanh,
        signum,
        sine,
        rbf,
        exponential,
        sqrt_abs,
        unit_interval_indicator,
        square,
        absolute,
    ]
    activations += [random_fourier_activation] * 10
    return random.choice(activations)


In [12]:
def sample_seq_len(min_seq: int, max_seq: int, log: bool = False) -> int:
    if min_seq >= max_seq:
        return max_seq
    if log:
        return int(np.exp(np.random.uniform(np.log(min_seq), np.log(max_seq))))
    return random.randint(min_seq, max_seq)


def sample_train_size(min_train: float, max_train: float, seq_len: int) -> int:
    if isinstance(min_train, float) and isinstance(max_train, float):
        low = int(min_train * seq_len)
        high = int(max_train * seq_len)
    else:
        low, high = int(min_train), int(max_train)
    if low >= high:
        return low
    return random.randint(low, high)


def x_sampler(seq_len: int, num_causes: int, sampling: str = "normal") -> np.ndarray:
    if sampling == "uniform":
        return np.random.rand(seq_len, num_causes)

    if sampling == "mixed":
        X = np.empty((seq_len, num_causes), dtype=float)
        for j in range(num_causes):
            choice = random.random()
            if choice < 0.5:
                X[:, j] = np.random.randn(seq_len)
            elif choice < 0.8:
                X[:, j] = np.random.rand(seq_len)
            else:
                cats = np.random.choice(4, size=seq_len)
                X[:, j] = cats / 3.0
        return X

    return np.random.randn(seq_len, num_causes)


In [13]:
from typing import Union

class SyntheticMLPSCM:
    """
    Simplified MLP-based SCM similar to TabICL's MLPSCM.
    Supports:
      - a single activation function shared across layers, OR
      - a list of activation functions (one per hidden layer)
    """

    def __init__(
        self,
        seq_len: int,
        num_features: int,
        num_outputs: int,
        is_causal: bool,
        num_causes: int,
        y_is_effect: bool,
        in_clique: bool,
        sort_features: bool,
        num_layers: int,
        hidden_dim: int,
        mlp_activation_fn: Union[
            Callable[[np.ndarray], np.ndarray],
            List[Callable[[np.ndarray], np.ndarray]],
        ],
        noise_std: float,
        sampling: str = "normal",
    ):
        self.seq_len = seq_len
        self.num_features = num_features
        self.num_outputs = num_outputs
        self.is_causal = is_causal
        self.num_causes = num_causes
        self.y_is_effect = y_is_effect
        self.in_clique = in_clique
        self.sort_features = sort_features
        self.num_layers = max(2, num_layers)
        self.hidden_dim = max(hidden_dim, num_outputs + 2 * num_features)
        self.noise_std = noise_std
        self.sampling = sampling

        # First layer is linear-only; remaining are (activation -> linear -> noise).
        num_hidden_layers = self.num_layers - 1

        if isinstance(mlp_activation_fn, list):
            if len(mlp_activation_fn) != num_hidden_layers:
                raise ValueError(f"Expected {num_hidden_layers} activations, got {len(mlp_activation_fn)}")
            self.activations = mlp_activation_fn
        else:
            self.activations = [mlp_activation_fn] * num_hidden_layers

        # initialize random weight matrices for each layer
        self.weights = []
        self.biases = []
        input_dim = num_causes
        for layer_idx in range(self.num_layers):
            output_dim = self.hidden_dim if layer_idx < self.num_layers - 1 else self.num_outputs
            w = np.random.randn(input_dim, output_dim) * (1.0 / math.sqrt(input_dim))
            b = np.zeros(output_dim)
            self.weights.append(w)
            self.biases.append(b)
            input_dim = output_dim

    def __call__(self) -> Tuple[np.ndarray, np.ndarray]:
        causes = x_sampler(self.seq_len, self.num_causes, sampling=self.sampling)

        outputs = [causes]
        x = causes

        # first linear layer without activation
        x = x @ self.weights[0] + self.biases[0]
        x += np.random.randn(*x.shape) * self.noise_std
        outputs.append(x)

        # remaining: activation -> linear -> noise
        for i in range(1, self.num_layers):
            activation_fn = self.activations[i - 1]
            x = activation_fn(x)
            x = x @ self.weights[i] + self.biases[i]
            x += np.random.randn(*x.shape) * self.noise_std
            outputs.append(x)

        outputs = outputs[2:]  # drop causes + first linear layer output
        return self.handle_outputs(causes, outputs)

    def handle_outputs(self, causes: np.ndarray, outputs: List[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
        if self.is_causal:
            outputs_flat = np.concatenate(outputs, axis=1)
            total_dim = outputs_flat.shape[1]

            if self.in_clique:
                start = random.randint(0, total_dim - self.num_outputs - self.num_features)
                perm = start + np.random.permutation(self.num_outputs + self.num_features)
            else:
                perm = np.random.permutation(total_dim - 1)

            indices_X = perm[self.num_outputs : self.num_outputs + self.num_features]
            if self.y_is_effect:
                indices_y = np.arange(total_dim - self.num_outputs, total_dim)
            else:
                indices_y = perm[: self.num_outputs]

            if self.sort_features:
                indices_X = np.sort(indices_X)

            X = outputs_flat[:, indices_X]
            y = outputs_flat[:, indices_y]
        else:
            X = causes
            y = outputs[-1]

        if y.ndim > 1 and y.shape[1] == 1:
            y = y[:, 0]
        return X, y


In [14]:
class SyntheticTreeSCM:
    """
    Simplified tree-based SCM using XGBoost for non-linear transformations.
    """

    def __init__(
        self,
        seq_len: int,
        num_features: int,
        num_outputs: int,
        is_causal: bool,
        num_causes: int,
        y_is_effect: bool,
        in_clique: bool,
        sort_features: bool,
        num_layers: int,
        hidden_dim: int,
        tree_depth_lambda: float,
        tree_n_estimators_lambda: float,
        noise_std: float,
        sampling: str = "normal",
    ):
        self.seq_len = seq_len
        self.num_features = num_features
        self.num_outputs = num_outputs
        self.is_causal = is_causal
        self.num_causes = num_causes
        self.y_is_effect = y_is_effect
        self.in_clique = in_clique
        self.sort_features = sort_features
        self.num_layers = max(2, num_layers)
        self.hidden_dim = max(hidden_dim, num_outputs + 2 * num_features)
        self.tree_depth_lambda = tree_depth_lambda
        self.tree_n_estimators_lambda = tree_n_estimators_lambda
        self.noise_std = noise_std
        self.sampling = sampling

    def __call__(self) -> Tuple[np.ndarray, np.ndarray]:
        causes = x_sampler(self.seq_len, self.num_causes, sampling=self.sampling)
        outputs = [causes]
        x = causes

        x = x + np.random.randn(*x.shape) * self.noise_std
        outputs.append(x)

        for i in range(self.num_layers):
            out_dim = self.hidden_dim if i < self.num_layers - 1 else self.num_outputs

            max_depth = 2 + int(np.random.exponential(1.0 / self.tree_depth_lambda))
            n_estimators = 1 + int(np.random.exponential(1.0 / self.tree_n_estimators_lambda))
            max_depth = min(max_depth, 4)
            n_estimators = min(n_estimators, 4)

            model = XGBRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                tree_method="hist",
                multi_strategy="multi_output_tree",
            )

            y_fake = np.random.randn(self.seq_len, out_dim)
            model.fit(x, y_fake)
            x = model.predict(x)
            x = x + np.random.randn(*x.shape) * self.noise_std
            outputs.append(x)

        outputs = outputs[2:]
        return self.handle_outputs(causes, outputs)

    def handle_outputs(self, causes: np.ndarray, outputs: List[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
        if self.is_causal:
            outputs_flat = np.concatenate(outputs, axis=1)
            total_dim = outputs_flat.shape[1]

            if self.in_clique:
                start = random.randint(0, total_dim - self.num_outputs - self.num_features)
                perm = start + np.random.permutation(self.num_outputs + self.num_features)
            else:
                perm = np.random.permutation(total_dim - 1)

            indices_X = perm[self.num_outputs : self.num_outputs + self.num_features]
            if self.y_is_effect:
                indices_y = np.arange(total_dim - self.num_outputs, total_dim)
            else:
                indices_y = perm[: self.num_outputs]

            if self.sort_features:
                indices_X = np.sort(indices_X)

            X = outputs_flat[:, indices_X]
            y = outputs_flat[:, indices_y]
        else:
            X = causes
            y = outputs[-1]

        if y.ndim > 1 and y.shape[1] == 1:
            y = y[:, 0]
        return X, y


In [15]:
def standardize(X: np.ndarray) -> np.ndarray:
    return StandardScaler().fit_transform(X)


def remove_outliers(X: np.ndarray, threshold: float = 4.0) -> np.ndarray:
    mean = np.nanmean(X, axis=0)
    std = np.nanstd(X, axis=0)
    std[std < 1e-6] = 1.0
    return np.clip(X, mean - threshold * std, mean + threshold * std)


def num2cat(X: np.ndarray, cat_prob: float = 0.2, max_categories: int = 10) -> np.ndarray:
    if random.random() < cat_prob:
        col_prob = random.random()
        for j in range(X.shape[1]):
            if random.random() < col_prob:
                num_cats = min(max(int(random.gammavariate(1.0, 10.0)), 2), max_categories)
                quantiles = np.quantile(X[:, j], q=np.linspace(0, 1, num_cats + 1)[1:-1])
                X[:, j] = np.digitize(X[:, j], quantiles)
                X[:, j] = X[:, j] / (num_cats - 1)
    return X


def permute_classes(labels: np.ndarray) -> np.ndarray:
    uniq = np.unique(labels)
    if len(uniq) <= 1:
        return labels
    perm = np.random.permutation(len(uniq))
    mapping = {u: perm[i] for i, u in enumerate(uniq)}
    return np.array([mapping[l] for l in labels])


def balanced_binarize(y: np.ndarray) -> np.ndarray:
    return (y > np.median(y)).astype(float)


def assign_multiclass(y: np.ndarray, num_classes: int, ordered_prob: float = 0.2) -> np.ndarray:
    boundaries = np.quantile(y, q=np.linspace(0, 1, num_classes + 1)[1:-1])
    labels = np.digitize(y, boundaries)
    if random.random() >= ordered_prob:
        labels = permute_classes(labels)
    return labels


def reg2cls(
    X: np.ndarray,
    y: np.ndarray,
    num_classes: int,
    max_features: int,
    balanced: bool = False,
    multiclass_ordered_prob: float = 0.0,
    cat_prob: float = 0.2,
    max_categories: int = 10,
    permute_features: bool = True,
    permute_labels: bool = True,
    scale_by_max_features: bool = False,
) -> Tuple[np.ndarray, np.ndarray]:
    X = num2cat(X, cat_prob=cat_prob, max_categories=max_categories)
    X = remove_outliers(X)
    X = standardize(X)

    if permute_features:
        X = X[:, np.random.permutation(X.shape[1])]

    if scale_by_max_features:
        X = X / (X.shape[1] / max_features)

    if X.shape[1] < max_features:
        X = np.concatenate([X, np.zeros((X.shape[0], max_features - X.shape[1]))], axis=1)

    y = (y - np.mean(y)) / (np.std(y) + 1e-6)

    if num_classes == 2 and balanced:
        y = balanced_binarize(y)
    else:
        y = assign_multiclass(y, num_classes, ordered_prob=multiclass_ordered_prob)

    if permute_labels:
        y = permute_classes(y)

    return X.astype(float), y.astype(float)


In [16]:
class SyntheticPriorGenerator:
    def __init__(
        self,
        batch_size: int = 1,
        min_features: int = 2,
        max_features: int = 10,
        max_classes: int = 5,
        min_seq_len: int = 32,
        max_seq_len: int = 1024,
        log_seq_len: bool = False,
        min_train_size: float = 0.1,
        max_train_size: float = 0.9,
        mix_probs: Tuple[float, float] = (0.7, 0.3),  # (mlp, tree)
    ):
        self.batch_size = batch_size
        self.min_features = min_features
        self.max_features = max_features
        self.max_classes = max_classes
        self.min_seq_len = min_seq_len
        self.max_seq_len = max_seq_len
        self.log_seq_len = log_seq_len
        self.min_train_size = min_train_size
        self.max_train_size = max_train_size
        self.mix_probs = mix_probs

    def sample_dataset_params(self) -> dict:
        seq_len = sample_seq_len(self.min_seq_len, self.max_seq_len, log=self.log_seq_len)
        train_size = sample_train_size(self.min_train_size, self.max_train_size, seq_len)
        num_features = random.randint(self.min_features, self.max_features)

        num_classes = random.randint(2, self.max_classes) if random.random() > 0.5 else 2
        prior_type = "mlp" if random.random() < self.mix_probs[0] else "tree"

        is_causal = random.choice([True, False])
        num_causes = max(1, int(np.random.lognormal(mean=1.5, sigma=0.5)))
        y_is_effect = random.choice([True, False])
        in_clique = random.choice([True, False])
        sort_features = random.choice([True, False])

        num_layers = max(2, int(np.random.lognormal(mean=2.5, sigma=0.5)))
        hidden_dim = max(4, int(np.random.lognormal(mean=3.5, sigma=0.8)))

        noise_std = 0.01 * 10 ** np.random.uniform(-2, 0)
        sampling = random.choice(["normal", "uniform", "mixed"])
        cat_prob = 0.2

        return dict(
            seq_len=seq_len,
            train_size=train_size,
            num_features=num_features,
            num_classes=num_classes,
            prior_type=prior_type,
            is_causal=is_causal,
            num_causes=num_causes,
            y_is_effect=y_is_effect,
            in_clique=in_clique,
            sort_features=sort_features,
            num_layers=num_layers,
            hidden_dim=hidden_dim,
            noise_std=noise_std,
            tree_depth_lambda=0.5,
            tree_n_estimators_lambda=0.5,
            sampling=sampling,
            cat_prob=cat_prob,
        )

    def generate_dataset(self, params: dict) -> Tuple[np.ndarray, np.ndarray, int]:
        while True:
            if params["prior_type"] == "mlp":
                # TabICL-like: 50% same activation across hidden layers, else per-layer.
                L = max(2, params["num_layers"])
                num_hidden_layers = L - 1
                if random.random() < 0.5:
                    act = sample_activation()
                    activation_fns = [act] * num_hidden_layers
                else:
                    activation_fns = [sample_activation() for _ in range(num_hidden_layers)]

                scm = SyntheticMLPSCM(
                    seq_len=params["seq_len"],
                    num_features=params["num_features"],
                    num_outputs=1,
                    is_causal=params["is_causal"],
                    num_causes=params["num_causes"],
                    y_is_effect=params["y_is_effect"],
                    in_clique=params["in_clique"],
                    sort_features=params["sort_features"],
                    num_layers=L,
                    hidden_dim=params["hidden_dim"],
                    mlp_activation_fn=activation_fns,
                    noise_std=params["noise_std"],
                    sampling=params["sampling"],
                )
                X, y = scm()
            else:
                scm = SyntheticTreeSCM(
                    seq_len=params["seq_len"],
                    num_features=params["num_features"],
                    num_outputs=1,
                    is_causal=params["is_causal"],
                    num_causes=params["num_causes"],
                    y_is_effect=params["y_is_effect"],
                    in_clique=params["in_clique"],
                    sort_features=params["sort_features"],
                    num_layers=params["num_layers"],
                    hidden_dim=params["hidden_dim"],
                    tree_depth_lambda=params["tree_depth_lambda"],
                    tree_n_estimators_lambda=params["tree_n_estimators_lambda"],
                    noise_std=params["noise_std"],
                    sampling=params["sampling"],
                )
                X, y = scm()

            X_proc, y_proc = reg2cls(
                X, y,
                num_classes=params["num_classes"],
                max_features=params["num_features"],
                cat_prob=params["cat_prob"],
                max_categories=10,
                permute_features=True,
                permute_labels=True,
                scale_by_max_features=False,
            )

            mask = np.std(X_proc, axis=0) > 1e-8
            if mask.sum() == 0:
                continue

            X_proc = X_proc[:, mask]
            uniq = np.unique(y_proc)
            if len(uniq) < 2:
                continue

            train_end = params["train_size"]
            if not (set(np.unique(y_proc[:train_end])) == set(np.unique(y_proc[train_end:])) == set(uniq)):
                continue

            return X_proc, y_proc, int(mask.sum())

    def get_batch(self) -> List[Tuple[np.ndarray, np.ndarray, int, dict]]:
        out = []
        for _ in range(self.batch_size):
            p = self.sample_dataset_params()
            X, y, d = self.generate_dataset(p)
            out.append((X, y, d, p))
        return out


In [17]:
set_seed(0)

gen = SyntheticPriorGenerator(
    batch_size=2,
    max_features=10,
    max_seq_len=512,
    max_classes=5,
    mix_probs=(0.7, 0.3),
)

batch = gen.get_batch()

for i, (X, y, d, p) in enumerate(batch):
    print(f"Dataset {i}: X={X.shape}, classes={np.unique(y)}, active_features={d}, prior_type={p['prior_type']}")
    train_end = p["train_size"]
    print("  train_end:", train_end, "train unique:", np.unique(y[:train_end]), "test unique:", np.unique(y[train_end:]))


Dataset 0: X=(464, 10), classes=[0. 1.], active_features=10, prior_type=tree
  train_end: 243 train unique: [0. 1.] test unique: [0. 1.]
Dataset 1: X=(290, 2), classes=[0. 1.], active_features=2, prior_type=mlp
  train_end: 64 train unique: [0. 1.] test unique: [0. 1.]
